#### Importing the training and testing datasets

In [1]:
import pandas as pd

In [2]:
X_train = pd.read_csv("../../data/heart/processed/X_train.csv")
X_test = pd.read_csv("../../data/heart/processed/X_test.csv")
y_train = pd.read_csv("../../data/heart/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../../data/heart/processed/y_test.csv").squeeze()

#### Logistic regression model training

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

logistic_regression_model = LogisticRegression(max_iter=100)

logistic_regression_model.fit(X_train,y_train)

y_pred_log = logistic_regression_model.predict(X_test)

print("LOGISTIC REGRESSION : ")
print("--------------------------------------------------------------------------------------------------------")
print(f"Accuracy : {accuracy_score(y_test,y_pred_log)} %")
print("--------------------------------------------------------------------------------------------------------")
print("Confusion Matrix : ")
print(f"{confusion_matrix(y_test,y_pred_log)}")
print("--------------------------------------------------------------------------------------------------------")
print("Classification Report : ")
print(f"{classification_report(y_test,y_pred_log)}")
print("--------------------------------------------------------------------------------------------------------")

LOGISTIC REGRESSION : 
--------------------------------------------------------------------------------------------------------
Accuracy : 0.832 %
--------------------------------------------------------------------------------------------------------
Confusion Matrix : 
[[129  19]
 [ 23  79]]
--------------------------------------------------------------------------------------------------------
Classification Report : 
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       148
           1       0.81      0.77      0.79       102

    accuracy                           0.83       250
   macro avg       0.83      0.82      0.82       250
weighted avg       0.83      0.83      0.83       250

--------------------------------------------------------------------------------------------------------


#### Random Forest model training

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

random_forest_model = RandomForestClassifier(n_estimators=50,random_state=42)

random_forest_model.fit(X_train,y_train)

y_pred_ran = random_forest_model.predict(X_test)

print("RANDOM FOREST : ")
print("--------------------------------------------------------------------------------------------------------")
print(f"Accuracy : {accuracy_score(y_test,y_pred_ran)} %")
print("--------------------------------------------------------------------------------------------------------")
print("Confusion Matrix : ")
print(f"{confusion_matrix(y_test,y_pred_ran)}")
print("--------------------------------------------------------------------------------------------------------")
print("Classification Report : ")
print(f"{classification_report(y_test,y_pred_ran)}")
print("--------------------------------------------------------------------------------------------------------")

RANDOM FOREST : 
--------------------------------------------------------------------------------------------------------
Accuracy : 1.0 %
--------------------------------------------------------------------------------------------------------
Confusion Matrix : 
[[148   0]
 [  0 102]]
--------------------------------------------------------------------------------------------------------
Classification Report : 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       148
           1       1.00      1.00      1.00       102

    accuracy                           1.00       250
   macro avg       1.00      1.00      1.00       250
weighted avg       1.00      1.00      1.00       250

--------------------------------------------------------------------------------------------------------


#### Gradient Boosting model training

In [14]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

gradient_boosting_model = GradientBoostingClassifier()

gradient_boosting_model.fit(X_train,y_train)

y_pred_boost = gradient_boosting_model.predict(X_test)

print("GRADIENT BOOST : ")
print("--------------------------------------------------------------------------------------------------------")
print(f"Accuracy : {accuracy_score(y_test,y_pred_boost)} %")
print("--------------------------------------------------------------------------------------------------------")
print("Confusion Matrix : ")
print(f"{confusion_matrix(y_test,y_pred_boost)}")
print("--------------------------------------------------------------------------------------------------------")
print("Classification Report : ")
print(f"{classification_report(y_test,y_pred_boost)}")
print("--------------------------------------------------------------------------------------------------------")

GRADIENT BOOST : 
--------------------------------------------------------------------------------------------------------
Accuracy : 1.0 %
--------------------------------------------------------------------------------------------------------
Confusion Matrix : 
[[148   0]
 [  0 102]]
--------------------------------------------------------------------------------------------------------
Classification Report : 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       148
           1       1.00      1.00      1.00       102

    accuracy                           1.00       250
   macro avg       1.00      1.00      1.00       250
weighted avg       1.00      1.00      1.00       250

--------------------------------------------------------------------------------------------------------


#### Cross validation scores checking

In [15]:
from sklearn.model_selection import cross_val_score

random_forest_scores = cross_val_score(random_forest_model, X_train, y_train, cv=5)
print("Random Forest CV Scores:", random_forest_scores)
print("Mean CV Accuracy:", random_forest_scores.mean())
print("---------------------------------------------------------------------------------------------")

logistic_regression_scores = cross_val_score(logistic_regression_model, X_train, y_train, cv=5)
print("Logistic Regression CV Scores:", logistic_regression_scores)
print("Mean CV Accuracy:", logistic_regression_scores.mean())
print("---------------------------------------------------------------------------------------------")

gradient_boost_scores = cross_val_score(gradient_boosting_model, X_train, y_train, cv=5)
print("Gradient Boost CV Scores:", gradient_boost_scores)
print("Mean CV Accuracy:", gradient_boost_scores.mean())
print("---------------------------------------------------------------------------------------------")

Random Forest CV Scores: [0.99333333 1.         1.         1.         1.        ]
Mean CV Accuracy: 0.9986666666666666
---------------------------------------------------------------------------------------------
Logistic Regression CV Scores: [0.85333333 0.9        0.88       0.82       0.88666667]
Mean CV Accuracy: 0.868
---------------------------------------------------------------------------------------------
Gradient Boost CV Scores: [1. 1. 1. 1. 1.]
Mean CV Accuracy: 1.0
---------------------------------------------------------------------------------------------


#### Testing the models : LR and RF

In [19]:
import pandas as pd
import joblib

input_data = pd.DataFrame([{
    "Age": 75,
    "Gender": "Female",
    "Cholesterol": 228,
    "Blood Pressure": 119,
    "Heart Rate": 66,
    "Smoking": "Current",
    "Alcohol Intake": "Heavy",
    "Exercise Hours": 1,
    "Family History": "No",
    "Diabetes": "No",
    "Obesity": "Yes",
    "Stress Level": 8,
    "Blood Sugar": 119,
    "Exercise Induced Angina": "Yes",
    "Chest Pain Type": "Atypical Angina"
}])

gender_map = {'Female':0,'Male':1} 
binary_map = {'Yes':1,'No':0}
bin_cols = ['Family History','Diabetes','Obesity','Exercise Induced Angina']
one_hot_cols = ['Smoking','Alcohol Intake','Chest Pain Type']
num_cols = ['Age','Cholesterol','Blood Pressure','Heart Rate','Exercise Hours','Stress Level','Blood Sugar']

input_data["Gender"] = input_data["Gender"].map(gender_map)
for col in bin_cols:
    input_data[col] = input_data[col].map(binary_map)
encoder = joblib.load("../../models/heart/one_hot_encoder.pkl")
encoded_array = encoder.transform(input_data[one_hot_cols])
encoded_df = pd.DataFrame(encoded_array,columns=encoder.get_feature_names_out(one_hot_cols))
input_data = input_data.drop(columns=one_hot_cols)
input_data = pd.concat([input_data,encoded_df], axis=1)
scaler = joblib.load("../../models/heart/standard_scaler.pkl")
input_data[num_cols] = scaler.transform(input_data[num_cols])

print(f"Logistic Regression : {logistic_regression_model.predict(input_data)}")
print(f"Random Forest : {random_forest_model.predict(input_data)}")

Logistic Regression : [1]
Random Forest : [1]


#### Saving models

In [20]:
joblib.dump(logistic_regression_model,"../../models/heart/logistic_regression_model.pkl")
joblib.dump(random_forest_model,"../../models/heart/random_forest_model.pkl")

['../../models/heart/random_forest_model.pkl']